# 0. Problem
## 185. Department Top Three Salaries — Hard
For each department, return employees whose salary is among the top three **distinct** salaries. Include ties.

Official: https://leetcode.com/problems/department-top-three-salaries/

# 1. Setup

In [ ]:
import pandas as pd
employee_rows=[(1,"Joe",85000,1),(2,"Henry",80000,2),(3,"Sam",60000,2),(4,"Max",90000,1),(5,"Janet",69000,1),(6,"Randy",85000,1),(7,"Will",70000,1),(8,"Amy",60000,2),(9,"Bob",55000,2),(10,"Cara",50000,2)]
department_rows=[(1,"IT"),(2,"Sales")]
employee_pd=pd.DataFrame(employee_rows,columns=["id","name","salary","departmentId"])
department_pd=pd.DataFrame(department_rows,columns=["id","name"])
employee_pd,department_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark=SparkSession.builder.getOrCreate()
employee_spark=spark.createDataFrame(employee_rows,["id","name","salary","departmentId"])
department_spark=spark.createDataFrame(department_rows,["id","name"])
employee_spark.createOrReplaceTempView("Employee")
department_spark.createOrReplaceTempView("Department")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
WITH ranked AS (
  SELECT id,name,salary,departmentId,
         DENSE_RANK() OVER(PARTITION BY departmentId ORDER BY salary DESC) AS salary_rank
  FROM Employee
)
SELECT d.name AS Department,r.name AS Employee,r.salary AS Salary
FROM ranked r
JOIN Department d ON r.departmentId=d.id
WHERE r.salary_rank<=3
ORDER BY Department,Salary DESC,Employee
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
ranked_pd=employee_pd.copy()
ranked_pd["salary_rank"]=ranked_pd.groupby("departmentId")["salary"].rank(method="dense",ascending=False)
top_pd=ranked_pd.loc[ranked_pd["salary_rank"]<=3]
result_pd=(top_pd.merge(department_pd,left_on="departmentId",right_on="id",suffixes=("_employee","_department")).rename(columns={"name_department":"Department","name_employee":"Employee","salary":"Salary"})[["Department","Employee","Salary"]].sort_values(["Department","Salary","Employee"],ascending=[True,False,True]).reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
w=Window.partitionBy("departmentId").orderBy(F.col("salary").desc())
ranked=employee_spark.withColumn("salary_rank",F.dense_rank().over(w)).filter(F.col("salary_rank")<=3)
result_spark=(ranked.alias("e").join(department_spark.alias("d"),F.col("e.departmentId")==F.col("d.id"),"inner").select(F.col("d.name").alias("Department"),F.col("e.name").alias("Employee"),F.col("e.salary").alias("Salary")).orderBy(F.asc("Department"),F.desc("Salary"),F.asc("Employee")))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| distinct salary rank | `DENSE_RANK()` | `.rank(method="dense")` | `F.dense_rank()` |
| per department | `PARTITION BY` | `.groupby()` | `Window.partitionBy()` |
| top 3 incl. ties | `rank<=3` | boolean filter | `.filter()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Employee, Department

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: employee_pd, department_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: employee_spark, department_spark